# How do you score a match between two lists of numbers?

Your music app just played you a song you love, by a band you have never heard of. Somewhere in a data center, two lists of numbers got compared, and the comparison said: same taste.

This notebook builds that comparison from scratch, in NumPy, and proves what it actually measures.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
print("numpy ready")

## A taste is an arrow

Strip it down. A taste is an arrow: each direction is a quality, and the length is how hard the song leans that way. Take one probe arrow (that's you) and four candidate arrows (four songs).

In [ ]:
probe = np.array([1.0, 3.0])
candidates = {
    "song_a": np.array([2.0, 6.0]),
    "song_b": np.array([3.0, -1.0]),
    "song_c": np.array([-2.0, -6.0]),
    "song_d": np.array([0.0, 2.0]),
}

print("probe:", probe)
for name, vec in candidates.items():
    print(name, vec)

Before you run this: the rule for scoring a match is multiply entry by entry, then add. Call it now, which candidate ends up on top, and does any score go negative?

In [ ]:
def dot_by_hand(a, b):
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

scores = {name: dot_by_hand(probe, vec) for name, vec in candidates.items()}
for name, s in scores.items():
    print(name, s)

for name, vec in candidates.items():
    assert abs(dot_by_hand(probe, vec) - np.dot(probe, vec)) < 1e-9

song_a comes out on top with the highest score. song_b lands at exactly zero. song_c goes negative. Multiply-and-add can drop below zero without hesitation. Similarity has a basement.

## Turn the probe and feel it

Before you run this: the candidates never move. Only the probe's direction changes. Predict: do the scores change anyway, even though nothing about the songs changed?

In [ ]:
def rotate(v, theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])

probe_turned = rotate(probe, np.pi / 2)
scores_turned = {name: dot_by_hand(probe_turned, vec) for name, vec in candidates.items()}

print("probe was:", probe, "-> now:", probe_turned)
for name in candidates:
    print(name, "before:", scores[name], "after:", scores_turned[name])

assert any(abs(scores[name] - scores_turned[name]) > 1e-6 for name in candidates)

The candidates never moved. Only the probe turned. The scores answered anyway, so the score can't be reading what the songs *are*. It's reading how the two arrows agree in direction.

## Agreement is not identity

Before you run this: predict whether a candidate that is not identical to the probe can still score positive, and how a stretched copy of the probe compares to a genuinely different arrow.

In [ ]:
same_direction = probe * 4  # four times longer, same direction, definitely not identical
different_vector = np.array([3.0, 1.0])  # a genuinely different arrow

score_scaled = dot_by_hand(probe, same_direction)
score_different = dot_by_hand(probe, different_vector)

print("probe . (4x probe) =", score_scaled)
print("probe . (different arrow) =", score_different)

assert not np.array_equal(probe, same_direction)
assert score_scaled > 0
assert score_scaled > score_different

## Name the machine you just used

Point the probe along a candidate and the sum climbs. Point it across and the sum dies to zero. Point it against and the sum goes negative. One multiply and add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

Before you run this: predict the sign of each of the three readings below.

In [ ]:
a = np.array([3.0, 4.0])
along = a * 2                    # same direction, different length
across = np.array([-4.0, 3.0])   # perpendicular to a
against = -a                     # opposite direction

reading_along = dot_by_hand(a, along)
reading_across = dot_by_hand(a, across)
reading_against = dot_by_hand(a, against)

print("along:", reading_along)
print("across:", reading_across)
print("against:", reading_against)

assert reading_along > 0
assert abs(reading_across) < 1e-9
assert reading_against < 0

Three positions, three readings: agree, ignore, oppose. The dial only ever swings because of direction. That's the whole machine, and it has a symbol: a . b, multiply pairwise and add.

## Different aisle, same meter

Swap the songs for a job posting. Score yourself against what the role wants: multiply the matched entries, add them up, one number for the fit. Before you run this: predict whether the same multiply-and-add can score a job match just as well as a song match.

In [ ]:
# entries: [python_hours, sql_reps, nights_free]
you = np.array([30.0, 10.0, 2.0])
role_wants = np.array([25.0, 15.0, 1.0])

fit_score = dot_by_hand(you, role_wants)
print("job fit score:", fit_score)

assert fit_score == np.dot(you, role_wants)

Same operation, different aisle. Multiply-and-add never asked what the numbers meant. That's exactly why it travels.

## The honest look

Before you run this: predict what happens to the score when a candidate's direction never changes, but it gets stretched to be five times longer.

In [ ]:
def unit(v):
    return v / np.linalg.norm(v)

candidate_short = np.array([1.0, 1.0])
candidate_long = candidate_short * 5.0

score_short = dot_by_hand(a, candidate_short)
score_long = dot_by_hand(a, candidate_long)

print("direction short:", unit(candidate_short))
print("direction long:", unit(candidate_long))
print("score short:", score_short)
print("score long:", score_long)

assert np.allclose(unit(candidate_short), unit(candidate_long))
assert score_long > score_short

The direction never moved, the printout above proves it: the two unit vectors match. The score climbed anyway, just because the arrow got longer. A long, loud song would win on sheer size, not on agreement. A padded resume would win on sheer length.

## For the walk home

What is the cheapest fix that keeps the direction and forgets the size?

Try this next: take `candidate_long` above, divide it by its own length before scoring it against `a`. See what number you get, then stretch it even further and check whether the score still climbs.